In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp
dbutils.widgets.text("lat", "53.336647")
dbutils.widgets.text("lon", "10.536493")
dbutils.widgets.dropdown("language", "de", ["de", "en"])
dbutils.widgets.text("units", "metric")

In [0]:

lat = dbutils.widgets.get("lat")
lon = dbutils.widgets.get("lon")
api_key = dbutils.secrets.get(scope="prod-ingestion", key="opean-weather-default-key")
language = dbutils.widgets.get("language")
units = dbutils.widgets.get("units")

url = f"https://api.openweathermap.org/data/2.5/weather"
params = {
    "lat" : lat,
    "lon" : lon,
    "appid" : api_key,
    "lang" : language,
    "units" : units
}

response = requests.get(url, params=params)
data = response.json()

display(data)


In [0]:
import json
from pyspark.sql.functions import current_timestamp
import uuid
from pyspark.sql.functions import current_timestamp, lit

json_str = json.dumps(data)
batch_id = str(uuid.uuid4())

df = spark.createDataFrame(
    [(json_str,)],
    ["raw_json"]
).withColumn("ingestion_time", current_timestamp())

spark.sql("CREATE SCHEMA IF NOT EXISTS dev.weather")
df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("dev.weather.bronze_weather")

In [0]:
spark.sql("""
    ALTER TABLE dev.weather.bronze_weather
    SET TBLPROPERTIES (
        'source_system' = 'openweathermap',
        'layer' = 'bronze',
        'update_frequency' = 'daily'
    )
""")

spark.sql("""
    COMMENT ON TABLE dev.weather.bronze_weather IS
    'Rohdaten von OpenWeatherMap API. Ungeparster JSON-String.'
""")

spark.sql("""
    ALTER TABLE dev.weather.bronze_weather
    ALTER COLUMN raw_json COMMENT 'Originale API-Antwort als JSON-String';
""")  

spark.sql("""
    ALTER TABLE dev.weather.bronze_weather
    ALTER COLUMN ingestion_time COMMENT 'Zeitpunkt des API-Calls'
""")
